# 7. Transformers y Atención

En este notebook aprenderás los fundamentos de los modelos Transformer y el mecanismo de atención, con ejemplos prácticos usando HuggingFace Transformers.

## Objetivo
- Comprender la teoría y arquitectura de los modelos Transformer.
- Implementar y usar modelos preentrenados con HuggingFace.
- Visualizar la atención para interpretar el modelo.
- Comparar modelos de diferentes tamaños.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado el [notebook 06 (RNN/LSTM)](./06_redes_recurrentes_rnn_lstm.ipynb).

- Conceptos de redes neuronales y procesamiento de secuencias.

> ⚠️ **Nota:** Este notebook usa tanto TensorFlow como **PyTorch** (vía HuggingFace). Asegúrate de tener instalados: `transformers`, `torch`.

## 1. Introducción teórica

El modelo Transformer revolucionó el procesamiento de secuencias gracias al mecanismo de atención.

- **Atención (Attention):** Calcula pesos para cada elemento de la secuencia.
- **Self-attention:** Cada elemento se relaciona consigo mismo y con los demás.
- **Multi-head attention:** Múltiples cabezas capturan diferentes relaciones.

**Arquitectura del Transformer:**
- Capas de atención multi-cabeza + normalización + feed-forward + residual.

| Modelo | Tipo | Uso principal |
|--------|------|---------------|
| **BERT** | Encoder | Clasificación, NER, QA |
| **GPT** | Decoder | Generación de texto |
| **T5** | Encoder-Decoder | Traducción, resumen |

**Ventajas:** Paralelización eficiente, manejo de dependencias largas.  
**Desventajas:** Requiere muchos datos y cómputo, costoso para secuencias muy largas.

## 2. Importación de librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# HuggingFace Transformers (usa PyTorch internamente)
from transformers import pipeline, BertTokenizer, BertModel
import torch

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

## 3. Clasificación de texto con modelo preentrenado

Usaremos un modelo preentrenado (DistilBERT) para clasificación de sentimientos.

In [ ]:
classifier = pipeline('sentiment-analysis',
                      model='distilbert-base-uncased-finetuned-sst-2-english')

texts = [
    "I love machine learning!",
    "This movie was terrible...",
    "Transformers are revolutionizing NLP.",
    "The weather today is okay, nothing special."
]

results = classifier(texts)
for text, res in zip(texts, results):
    emoji = '😊' if res['label'] == 'POSITIVE' else '😞'
    print(f"{emoji} [{res['label']}: {res['score']:.2f}] {text}")

## 4. Visualización de la atención

Mostraremos cómo visualizar los pesos de atención de BERT.

In [ ]:
# Cargar modelo y tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, output_attentions=True)

sentence = "Transformers are changing deep learning."
inputs = tokenizer(sentence, return_tensors='pt')
outputs = model(**inputs)
attentions = outputs.attentions  # Lista de tensores (una por capa)

# Visualizar atención de la última capa, cabeza 0
att = attentions[-1][0, 0].detach().numpy()  # [tokens, tokens]
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

plt.figure(figsize=(8,6))
sns.heatmap(att, xticklabels=tokens, yticklabels=tokens, cmap='viridis')
plt.title('Matriz de atención (última capa, cabeza 0)')
plt.xlabel('Token atendido')
plt.ylabel('Token que atiende')
plt.show()

In [ ]:
# Visualizar múltiples cabezas de atención
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    if i < attentions[-1].shape[1]:  # número de cabezas
        att_head = attentions[-1][0, i].detach().numpy()
        sns.heatmap(att_head, xticklabels=tokens, yticklabels=tokens,
                    cmap='viridis', ax=ax, cbar=False)
        ax.set_title(f'Cabeza {i}')
    else:
        ax.axis('off')
plt.suptitle('Atención por cabeza (última capa)')
plt.tight_layout()
plt.show()

## 5. Comparación de modelos de diferentes tamaños

Comparemos la velocidad y las predicciones de modelos de diferente tamaño.

In [ ]:
import time

model_names = [
    'distilbert-base-uncased-finetuned-sst-2-english',
]

test_texts = [
    "This product is amazing and I love it!",
    "Terrible quality, worst purchase ever.",
    "It's okay, nothing special."
]

for model_name in model_names:
    clf = pipeline('sentiment-analysis', model=model_name)
    start = time.time()
    preds = clf(test_texts)
    elapsed = time.time() - start
    print(f'\n{model_name} ({elapsed:.2f}s):')
    for text, pred in zip(test_texts, preds):
        print(f'  {pred["label"]} ({pred["score"]:.2f}) → {text}')

### Recomendaciones prácticas para Transformers

| Aspecto | Recomendación |
|---------|---------------|
| **Modelos preentrenados** | Usa siempre que sea posible (HuggingFace Hub) |
| **Fine-tuning** | Requiere menos datos que entrenar desde cero |
| **Batch size** | Ajusta según la memoria GPU disponible |
| **Interpretación** | Visualiza la atención para entender el modelo |
| **GPU** | Esencial para entrenamiento; inferencia posible en CPU |
| **Modelos pequeños** | DistilBERT, TinyBERT para restricciones de cómputo |

## 6. Discusión y Conclusiones

- Los Transformers son el estado del arte en NLP y cada vez más en visión y audio.
- Los modelos preentrenados permiten resultados excelentes con poco código.
- Visualizar la atención ayuda a interpretar qué aprende el modelo.
- El tamaño del modelo impacta velocidad vs calidad.
- En el siguiente notebook exploraremos GANs para generación de datos.

## 7. Ejercicios Propuestos

1. **Ejercicio 1:** Usa el pipeline de `zero-shot-classification` para clasificar textos en categorías personalizadas (deportes, tecnología, política).

2. **Ejercicio 2:** Prueba el pipeline de `text-generation` con GPT-2. Genera textos a partir de diferentes prompts.

3. **Ejercicio 3:** Compara la atención entre diferentes capas del modelo (primera vs última). ¿Qué patrones observas?

4. **Ejercicio 4 (Avanzado):** Haz fine-tuning de DistilBERT en un dataset personalizado de clasificación de texto usando HuggingFace Trainer.

## 8. Referencias y Recursos

- [HuggingFace Transformers](https://huggingface.co/docs/transformers/index)
- [HuggingFace Model Hub](https://huggingface.co/models)
- Vaswani et al. (2017). *Attention is All You Need.*
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)

---

📎 **Notebook anterior:** [06. Redes Recurrentes (RNN/LSTM)](./06_redes_recurrentes_rnn_lstm.ipynb)  
📎 **Notebook siguiente:** [08. Redes Generativas (GANs)](./08_gans.ipynb)